# 02 — Data quality

**Question:** is this extract fit for modelling, and what would block it?

Eight automated checks run on every pipeline execution. Each returns a severity rather than raising, so one pass surfaces every problem.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
from smartnet import config
from smartnet.data import loader, validation

df = loader.load_analysis_frame()
report = validation.run_all(df)
report

In [ ]:
print('Blocking failure:', validation.has_blocking_failure(report))

## Missingness

In [ ]:
miss = loader.describe_missingness(df)
print(miss.head(10).to_string(index=False))

## Label consistency

The nested schemes must agree. Every row that is *Nothing* in the binary scheme must be *Nothing* in all schemes, and 5-category Enter/Exit must collapse into 4-category *Enter or exit*. This is a real integrity test of the tagging process, not a formality.

In [ ]:
print(pd.crosstab(df.motion_5cat.map(config.LABEL_MAPS['motion_5cat']),
                  df.motion_4cat.map(config.LABEL_MAPS['motion_4cat'])).to_string())

## Class support

The rarest class carries 44 epochs. Under 5-fold cross-validation that is roughly 9 test examples per fold, so per-class estimates for Enter and Exit carry wide uncertainty. Flagged, not fixed — no amount of resampling creates information that is not there.

In [ ]:
for col in ['motion_4cat','motion_5cat']:
    vc = df[col].value_counts()
    names = config.LABEL_MAPS[col]
    print(f'{col}: rarest = {names[int(vc.idxmin())]} at {vc.min()} epochs '
          f'(~{vc.min()//config.N_SPLITS} per CV fold)')

## Deliberate non-decisions

Three things this pipeline does **not** do, each on purpose:

- **No outlier removal.** Large accelerations are the signal, not noise. Values are range-checked and reported, never clipped.
- **No imputation.** There is nothing missing.
- **No resampling of minority classes.** Class weighting is used inside the models instead, so reported support stays honest.